In [3]:
# import scanpy as sc
# import pandas as pd

# dataset_number = '7'
# adata =sc.read_h5ad(f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad")
# adata.obs['cell_type']=adata.obs['celltype_final']
# print(adata.obs['cell_type'].value_counts())
# adata.write_h5ad(f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/scRNA.h5ad")
# adata_st = sc.read_h5ad(f"/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset{dataset_number}/Spatial.h5ad")
# st_label = adata_st.uns['density']
# st_label_df = pd.DataFrame(st_label)
# st_label_df.to_csv(f"/mnt/d/ST_Graduation_Project/SC_MAP_ST/evaluate/Data{dataset_number}/dataset{dataset_number}_density.csv")

In [4]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset7/scRNA.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset7/Spatial.h5ad" \
    --n_epochs 150 \
    --output_dir ../deconv_results/DATA7
#    --marker_selection_method l1 
#    --precomputed_marker_file "../deconv_results/STARmap/final_genes.txt"

Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4
   Batch size: 1024
   Epochs: 150
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 128
   Loss type: MSE
   Lambda MMD: 0.05
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset7/scRNA.h5ad
   SC shape: (10000, 22066)
   SC avg counts/cell: 8137.5 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset7/Spatial.h5ad
   ST shape: (1000, 25199)
   Common genes: 17647
   SC final: (10000, 17647)
   ST final: (1000, 17647)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 58 clusters
Marker genes per cluster:
   0: 100 -> 100 (after correlation)
   1: 100 -> 100 (after correlation)
   10: 100 -> 100 (after correlation)
   11: 100 -> 100 (after correlatio

VAE Training: 100%|██████████| 150/150 [01:46<00:00,  1.41epoch/s, Train=423.4908, Recon=416.4559, KL=70.3491, MMD=0.0098, Test=399.8476] 


📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (10000, 1640)
   Number of clusters: 58
   Computed embeddings shape: (10000, 128)
Computing cluster centers with mean aggregation...
   Completed: 58 clusters with mean centers and expressions
Extracting celltype-cluster mapping (using 'cell_type' column)...
Visualizing modality alignment...
Generating UMAP visualization for modality alignment...
   Computing UMAP on 10890 samples with 128 dims...
   UMAP visualization saved to: ../deconv_results/DATA7/modality_alignment_umap.png
Saving model to: ../deconv_results/DATA7/final_vae.pth
   Average cell counts: 8137.5 (for Stage 2 scale factor)
✅ Saved VAE model (weights only): ../deconv_results/DATA7/final_vae.pth
Saving cluster data to: ../deconv_results/DATA7/final_vae_cluster_data.npz
   ✓ Cluster IDs: 58
   ✓ Prototypes: (58, 128)
   ✓ Expressions (marker): (58, 1640)
   ✓ Expressions (full): 58 clusters × 17647 genes
   ✓ Celltype mapping: 58 clusters
✅ 

In [5]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/SimualtedSpatalData/dataset7/Spatial.h5ad" \
    --stage1_model_path "../deconv_results/DATA7/final_vae.pth" \
    --output_dir "../deconv_results/DATA7/" \
    --n_epoch 250

Sample name: Spatial
Stage 1 model: ../deconv_results/DATA7/final_vae.pth
Output directory: ../deconv_results/DATA7/
Device: cpu
Weight threshold: 0.001
Loading pretrained VAE Encoder...
   VAE architecture: 1640 -> 128
   Output type: mse
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 10000 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/DATA7/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([58, 128])
   Cluster expressions (marker): torch.Size([58, 1640])
   Cluster expressions (all genes): 58 × 17647
   Loaded celltype mapping: 58 clusters
   Average cell counts: 8137.5
Loaded all genes list: 17647 genes
VAE Encoder loaded: 1640 -> 128
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50'

GAT Training: 100%|██████████| 250/250 [10:02<00:00,  2.41s/epoch, Total=2.1008, MSE=0.1600, Spot_Cosine=0.0928, Gene_Cosine=0.3419, Pearson=0.1221, Gene_Pearson=0.4807, P_pat=0, M_pat=10, C_pat=10] 


Evaluating model results...
Applying weight threshold: 0.001
   Non-zero elements: 30000 -> 19977 (34.4%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   ✅ Scale factor computed from MARKER genes (1640 genes)
      Spot marker total (real): mean=3603.9
      Reconstructed marker total: mean=898.6
      Scale factor: min=0.7541, max=13.4696, mean=3.9541
   Saved: ../deconv_results/DATA7//Spatial_reconstructed_all_genes.csv
   Cell type composition...
   Using cluster→celltype mapping from Stage 1 checkpoint (58 entries).
   Found duplicate celltype names: ['CD8_T', 'CD4_T', 'Monocytes', 'Endothelial', 'Macrophage', 'Mast', 'Muscle_cells', 'Basal', 'Alveolar_Type_2', 'Fibroblast', 'NK', 'Ciliated']. Merging corresponding cluster columns by summing weights.
   Columns before: 58, after merge: 19
   Saved cell composition (celltype): ../deconv_results/DATA7//Spatial_cell_composition.csv
   Saved cluster composition